In [ ]:
#Create batched dynamic graph dataset for GNN training

# Combined Correlation, Mutual Information, Alpha Power, and Beta Power into a dynamic graph with shape (85330, 14, 14, 4).
# Applied the predefined EEG topology mask and removed self-connections.
# Preserved the existing Train/Validation/Test split.
# Split the dynamic graph into batches of 256 samples due to Google Colab memory limitations.
# Saved Train, Validation, and Test batches separately.
# Created and saved matching Arousal, Valence, and Dominance label batches.
# Verified X/label alignment for all splits.
# Prepared the dataset for the next GNN/model implementation.

In [ ]:
#Note:
# Dataset Splitting and Batching
#
# Due to Google Colab's memory and computational limitations,
# the complete dynamic graph dataset cannot be processed
# efficiently as a single large array during model training.
#
# Therefore, the dataset was split into Train / Validation /
# Test sets and stored in separate batches of 256 samples.
# This allows the data to be loaded and processed batch by batch
# without requiring the entire dataset to be loaded into RAM.

In [1]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
from pathlib import Path

# Change this path if your project is stored somewhere else
DATA_DIR = Path(
    "/content/drive/MyDrive/EEG_project/processed_data"
)

print("Data directory:", DATA_DIR)

Mounted at /content/drive
Data directory: /content/drive/MyDrive/EEG_project/processed_data


In [4]:
from pathlib import Path

# Search for the processed_data folder in Google Drive
matches = list(Path("/content/drive/MyDrive").rglob("processed_data"))

for path in matches:
    print(path)

/content/drive/MyDrive/processed_data


In [5]:
# Load processed EEG features

import numpy as np
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/processed_data")

correlation = np.load(DATA_DIR / "correlation.npy")
mi = np.load(DATA_DIR / "mi.npy")

alpha_power = np.load(DATA_DIR / "alpha_power.npy")
beta_power = np.load(DATA_DIR / "beta_power.npy")

subjects = np.load(DATA_DIR / "subjects.npy")
trials = np.load(DATA_DIR / "trials.npy")

print("Correlation:", correlation.shape)
print("MI:", mi.shape)
print("Alpha Power:", alpha_power.shape)
print("Beta Power:", beta_power.shape)
print("Subjects:", subjects.shape)
print("Trials:", trials.shape)

Correlation: (85330, 14, 14)
MI: (85330, 14, 14)
Alpha Power: (85330, 14)
Beta Power: (85330, 14)
Subjects: (85330,)
Trials: (85330,)


In [6]:
# Apply topology mask to connectivity features

topology = np.array([
    [0,1,1,0,0,0,0,0,0,0,0,1,1,1],
    [1,0,1,1,1,0,0,0,0,0,0,0,1,0],
    [1,1,0,1,0,0,0,0,0,0,1,1,0,0],
    [0,1,1,0,1,1,0,0,0,0,1,0,0,0],
    [0,1,0,1,0,1,0,0,0,1,0,0,0,0],
    [0,0,0,1,1,0,1,0,1,0,0,0,0,0],
    [0,0,0,0,0,1,0,1,0,0,0,0,0,0],
    [0,0,0,0,0,0,1,0,1,0,0,0,0,0],
    [0,0,0,0,0,1,0,1,0,1,1,0,0,0],
    [0,0,0,0,1,0,0,0,1,0,1,0,0,0],
    [0,0,1,1,0,0,0,0,1,1,0,1,1,0],
    [1,0,1,0,0,0,0,0,0,0,1,0,1,1],
    [1,1,0,0,0,0,0,0,0,0,1,1,0,1],
    [1,0,0,0,0,0,0,0,0,0,0,1,1,0]
], dtype=np.float32)

np.fill_diagonal(topology, 0)

dynamic_corr = correlation * topology
dynamic_mi = mi * topology

print("Topology shape:", topology.shape)
print("Dynamic Correlation:", dynamic_corr.shape)
print("Dynamic MI:", dynamic_mi.shape)

Topology shape: (14, 14)
Dynamic Correlation: (85330, 14, 14)
Dynamic MI: (85330, 14, 14)


In [7]:
print("Correlation diagonal:",
      dynamic_corr[0].diagonal())

print("MI diagonal:",
      dynamic_mi[0].diagonal())

Correlation diagonal: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
MI diagonal: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [8]:
# Convert node-level Alpha/Beta Power to edge-level features
#
# Edge(i,j) = mean(Power(i), Power(j))
# Only topology-connected edges are retained.

num_windows = alpha_power.shape[0]
num_nodes = 14

dynamic_alpha_power = np.zeros(
    (num_windows, num_nodes, num_nodes),
    dtype=np.float32
)

dynamic_beta_power = np.zeros(
    (num_windows, num_nodes, num_nodes),
    dtype=np.float32
)

for i in range(num_nodes):
    for j in range(num_nodes):

        if topology[i, j] == 1:

            dynamic_alpha_power[:, i, j] = (
                alpha_power[:, i] +
                alpha_power[:, j]
            ) / 2

            dynamic_beta_power[:, i, j] = (
                beta_power[:, i] +
                beta_power[:, j]
            ) / 2

print("Dynamic Alpha Power:", dynamic_alpha_power.shape)
print("Dynamic Beta Power:", dynamic_beta_power.shape)

Dynamic Alpha Power: (85330, 14, 14)
Dynamic Beta Power: (85330, 14, 14)


In [14]:
# Combine the four dynamic graph features
#
# Channel 0 -> Correlation
# Channel 1 -> Mutual Information
# Channel 2 -> Alpha Power
# Channel 3 -> Beta Power
dynamic_graph = np.stack(
    [
        dynamic_corr,
        dynamic_mi,
        dynamic_alpha_power,
        dynamic_beta_power
    ],
    axis=-1
)

print("Dynamic Graph shape:", dynamic_graph.shape)

Dynamic Graph shape: (85330, 14, 14, 4)


In [15]:
# Save the final dynamic graph

OUTPUT_PATH = DATA_DIR / "dynamic_graph.npy"

np.save(
    OUTPUT_PATH,
    dynamic_graph.astype(np.float32)
)

print("Dynamic graph saved successfully.")
print("Path:", OUTPUT_PATH)
print("Shape:", dynamic_graph.shape)

file_size_gb = OUTPUT_PATH.stat().st_size / (1024 ** 3)
print(f"File size: {file_size_gb:.2f} GB")

Dynamic graph saved successfully.
Path: /content/drive/MyDrive/processed_data/dynamic_graph.npy
Shape: (85330, 14, 14, 4)
File size: 0.25 GB


In [16]:
# Split dynamic_graph using the existing train/val/test split
# and save the results in batches
from pathlib import Path

# Paths
DATA_DIR = Path("/content/drive/MyDrive/processed_data")

DYNAMIC_GRAPH_PATH = DATA_DIR / "dynamic_graph.npy"
SPLIT_DIR = DATA_DIR / "split_dataset"

OUTPUT_DIR = DATA_DIR / "dynamic_graf_split_dataset"

# Create output directories
TRAIN_DIR = OUTPUT_DIR / "X_train"
VAL_DIR = OUTPUT_DIR / "X_val"
TEST_DIR = OUTPUT_DIR / "X_test"

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
VAL_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: /content/drive/MyDrive/processed_data/dynamic_graf_split_dataset


In [17]:
# Load dynamic graph and existing split data
dynamic_graph = np.load(
    DYNAMIC_GRAPH_PATH,
    mmap_mode="r"
)

X_train = np.load(SPLIT_DIR / "X_train.npy")
X_val = np.load(SPLIT_DIR / "X_val.npy")
X_test = np.load(SPLIT_DIR / "X_test.npy")

subjects_train = np.load(SPLIT_DIR / "subjects_train.npy")
subjects_val = np.load(SPLIT_DIR / "subjects_val.npy")
subjects_test = np.load(SPLIT_DIR / "subjects_test.npy")

print("Dynamic graph:", dynamic_graph.shape)

print("Existing X_train:", X_train.shape)
print("Existing X_val:", X_val.shape)
print("Existing X_test:", X_test.shape)

Dynamic graph: (85330, 14, 14, 4)
Existing X_train: (59360, 224)
Existing X_val: (11130, 224)
Existing X_test: (14840, 224)


In [18]:
# Reconstruct train / validation / test indices
# using subject IDs from the existing split

subjects = np.load(DATA_DIR / "subjects.npy")

train_subjects = np.unique(subjects_train)
val_subjects = np.unique(subjects_val)
test_subjects = np.unique(subjects_test)

train_indices = np.where(np.isin(subjects, train_subjects))[0]
val_indices = np.where(np.isin(subjects, val_subjects))[0]
test_indices = np.where(np.isin(subjects, test_subjects))[0]

print("Train samples:", len(train_indices))
print("Validation samples:", len(val_indices))
print("Test samples:", len(test_indices))

print("Total:", len(train_indices) + len(val_indices) + len(test_indices))

Train samples: 59360
Validation samples: 11130
Test samples: 14840
Total: 85330


In [19]:
# Save dynamic graph splits in batches

BATCH_SIZE = 256


def save_batches(indices, output_dir, split_name):
    num_samples = len(indices)

    for batch_id, start in enumerate(
        range(0, num_samples, BATCH_SIZE)
    ):

        end = min(start + BATCH_SIZE, num_samples)

        batch_indices = indices[start:end]

        batch = np.asarray(
            dynamic_graph[batch_indices],
            dtype=np.float32
        )

        output_path = output_dir / f"{split_name}_{batch_id:03d}.npy"

        np.save(output_path, batch)

        print(f"{output_path.name} -> {batch.shape}")


save_batches(train_indices, TRAIN_DIR, "train")
save_batches(val_indices, VAL_DIR, "val")
save_batches(test_indices, TEST_DIR, "test")

print("\nAll batches saved successfully.")

train_000.npy -> (256, 14, 14, 4)
train_001.npy -> (256, 14, 14, 4)
train_002.npy -> (256, 14, 14, 4)
train_003.npy -> (256, 14, 14, 4)
train_004.npy -> (256, 14, 14, 4)
train_005.npy -> (256, 14, 14, 4)
train_006.npy -> (256, 14, 14, 4)
train_007.npy -> (256, 14, 14, 4)
train_008.npy -> (256, 14, 14, 4)
train_009.npy -> (256, 14, 14, 4)
train_010.npy -> (256, 14, 14, 4)
train_011.npy -> (256, 14, 14, 4)
train_012.npy -> (256, 14, 14, 4)
train_013.npy -> (256, 14, 14, 4)
train_014.npy -> (256, 14, 14, 4)
train_015.npy -> (256, 14, 14, 4)
train_016.npy -> (256, 14, 14, 4)
train_017.npy -> (256, 14, 14, 4)
train_018.npy -> (256, 14, 14, 4)
train_019.npy -> (256, 14, 14, 4)
train_020.npy -> (256, 14, 14, 4)
train_021.npy -> (256, 14, 14, 4)
train_022.npy -> (256, 14, 14, 4)
train_023.npy -> (256, 14, 14, 4)
train_024.npy -> (256, 14, 14, 4)
train_025.npy -> (256, 14, 14, 4)
train_026.npy -> (256, 14, 14, 4)
train_027.npy -> (256, 14, 14, 4)
train_028.npy -> (256, 14, 14, 4)
train_029.npy 

In [20]:
# Check saved batches

for split in ["X_train", "X_val", "X_test"]:
    split_dir = OUTPUT_DIR / split
    files = sorted(split_dir.glob("*.npy"))

    print(f"\n{split}")
    print("Number of batches:", len(files))

    if files:
        first_batch = np.load(files[0], mmap_mode="r")
        last_batch = np.load(files[-1], mmap_mode="r")

        print("First batch:", first_batch.shape)
        print("Last batch:", last_batch.shape)


X_train
Number of batches: 232
First batch: (256, 14, 14, 4)
Last batch: (224, 14, 14, 4)

X_val
Number of batches: 44
First batch: (256, 14, 14, 4)
Last batch: (122, 14, 14, 4)

X_test
Number of batches: 58
First batch: (256, 14, 14, 4)
Last batch: (248, 14, 14, 4)


In [22]:
# Save labels in a clean structure

import numpy as np
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/processed_data")
OUTPUT_DIR = DATA_DIR / "dynamic_graf_split_dataset"
LABEL_DIR = OUTPUT_DIR / "labels"

BATCH_SIZE = 256

# Load labels

arousal = np.load(DATA_DIR / "arousal.npy")
valence = np.load(DATA_DIR / "valence.npy")
dominance = np.load(DATA_DIR / "dominance.npy")

print("Labels loaded:")
print("Arousal:", arousal.shape)
print("Valence:", valence.shape)
print("Dominance:", dominance.shape)


# Create folders

for split in ["train", "val", "test"]:
    for label in ["arousal", "valence", "dominance"]:
        (LABEL_DIR / split / label).mkdir(
            parents=True,
            exist_ok=True
        )


# Save function
def save_batches(label_data, indices, split, label_name):

    output_dir = LABEL_DIR / split / label_name

    num_samples = len(indices)

    for batch_id, start in enumerate(
        range(0, num_samples, BATCH_SIZE)
    ):

        end = min(start + BATCH_SIZE, num_samples)

        batch_indices = indices[start:end]

        batch = label_data[batch_indices].astype(np.float32)

        output_path = output_dir / f"batch_{batch_id:03d}.npy"

        np.save(output_path, batch)


# Save all labels

for split, indices in [
    ("train", train_indices),
    ("val", val_indices),
    ("test", test_indices)
]:

    save_batches(arousal, indices, split, "arousal")
    save_batches(valence, indices, split, "valence")
    save_batches(dominance, indices, split, "dominance")


print("\nAll labels saved successfully.")
print("Location:", LABEL_DIR)

Labels loaded:
Arousal: (85330,)
Valence: (85330,)
Dominance: (85330,)

All labels saved successfully.
Location: /content/drive/MyDrive/processed_data/dynamic_graf_split_dataset/labels


In [23]:
# Final X / Label alignment check

for split in ["train", "val", "test"]:

    if split == "train":
        x_dir = OUTPUT_DIR / "X_train"
    elif split == "val":
        x_dir = OUTPUT_DIR / "X_val"
    else:
        x_dir = OUTPUT_DIR / "X_test"

    x_files = sorted(x_dir.glob("*.npy"))

    print(f"\n{split.upper()}")

    for i in [0, len(x_files) - 1]:

        x = np.load(x_files[i], mmap_mode="r")

        ar = np.load(
            LABEL_DIR / split / "arousal" /
            f"batch_{i:03d}.npy"
        )

        va = np.load(
            LABEL_DIR / split / "valence" /
            f"batch_{i:03d}.npy"
        )

        do = np.load(
            LABEL_DIR / split / "dominance" /
            f"batch_{i:03d}.npy"
        )

        print(f"Batch {i:03d}")
        print("X:", x.shape)
        print("Arousal:", ar.shape)
        print("Valence:", va.shape)
        print("Dominance:", do.shape)

        assert x.shape[0] == ar.shape[0]
        assert x.shape[0] == va.shape[0]
        assert x.shape[0] == do.shape[0]

print("\nAlignment check passed successfully.")


TRAIN
Batch 000
X: (256, 14, 14, 4)
Arousal: (256,)
Valence: (256,)
Dominance: (256,)
Batch 231
X: (224, 14, 14, 4)
Arousal: (224,)
Valence: (224,)
Dominance: (224,)

VAL
Batch 000
X: (256, 14, 14, 4)
Arousal: (256,)
Valence: (256,)
Dominance: (256,)
Batch 043
X: (122, 14, 14, 4)
Arousal: (122,)
Valence: (122,)
Dominance: (122,)

TEST
Batch 000
X: (256, 14, 14, 4)
Arousal: (256,)
Valence: (256,)
Dominance: (256,)
Batch 057
X: (248, 14, 14, 4)
Arousal: (248,)
Valence: (248,)
Dominance: (248,)

Alignment check passed successfully.
